In [ ]:
import copy
from heapq import heappush, heappop

# Giả sử puzzle(n=3)
n = 3

# 4 vị trí dịch chuyển tương ứng bottom, left, top, right
rows = [ 1, 0, -1, 0 ]
cols = [ 0, -1, 0, 1 ]

# Tạo một lớp hàng đợi ưu tiên
class priorityQueue:
    def __init__(self):
        self.heap = []

    # Inserting a new key 'key'
    def push(self, key):
        heappush(self.heap, key)

    # funct to remove the element that is min from the Priority Queue
    def pop(self):
        return heappop(self.heap)

    # funct to check if the Queue is empty or not
    def empty(self):
        if not self.heap:
            return True
        else:
            return False

# structure of the node
class nodes:
    def __init__(self, parent, mats, empty_tile_posi, costs, levels):
        self.parent = parent

        # Useful for Storing the matrix
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.costs = costs
        self.levels = levels

    def __lt__(self, nxt):
        return self.costs < nxt.costs

def calculateCosts(mats, final) -> int:
    count = 0
    for i in range(n):
        for j in range(n):
            if ((mats[i][j]) and (mats[i][j] != final[i][j])):
                count += 1
    return count

def newNodes(mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final) -> nodes:
    # Copying data from the parent matrixes to the present matrixes
    new_mats = copy.deepcopy(mats)

    # Moving the tile by 1 position
    x1 = empty_tile_posi[0]
    y1 = empty_tile_posi[1]
    x2 = new_empty_tile_posi[0]
    y2 = new_empty_tile_posi[1]

    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    # Setting the no. of misplaced tiles
    costs = calculateCosts(new_mats, final)

    new_nodes = nodes(parent, new_mats, new_empty_tile_posi, costs, levels)
    return new_nodes

# func to print the N by N matrix
def printMatsrix(mats):
    for i in range(n):
        for j in range(n):
            print("%d " % (mats[i][j]), end = " ")
        print()

# func to know if (x, y) is a valid or invalid matrix coordinates
def isSafe(x, y):
    return x >= 0 and x < n and y >= 0 and y < n

# Printing the path from the root node to the final node
def printPath(root):
    if root == None:
        return

    printPath(root.parent)
    printMatsrix(root.mats)
    print()

def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()

    # Creating the root node
    costs = calculateCosts(initial, final)
    root = nodes(None, initial, empty_tile_posi, costs, 0)

    # Adding root to the list of live nodes
    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()

        # If minimum.costs == 0, that means we found the solution
        if minimum.costs == 0:
            printPath(minimum)
            return

        # Generating all feasible children
        for i in range(n):
            new_tile_posi = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i],
            ]

            if isSafe(new_tile_posi[0], new_tile_posi[1]):
                # Creating a child node
                child = newNodes(minimum.mats,
                                 minimum.empty_tile_posi,
                                 new_tile_posi,
                                 minimum.levels + 1,
                                 minimum,
                                 final)
                # Adding the child to the list of live nodes
                pq.push(child)

# --- Khởi tạo dữ liệu và chạy chương trình ---
initial = [ [ 1, 2, 3 ],
            [ 5, 6, 0 ],
            [ 7, 8, 4 ] ]

final = [ [ 1, 2, 3 ],
          [ 5, 8, 6 ],
          [ 0, 7, 4 ] ]

empty_tile_posi = [1, 2]

solve(initial, empty_tile_posi, final)


1  2  3  
5  6  0  
7  8  4  

1  2  3  
5  0  6  
7  8  4  

1  2  3  
5  8  6  
7  0  4  

1  2  3  
5  8  6  
0  7  4  



In [ ]:
from heapq import heappush, heappop
import copy

n = 3

moves = [(1,0), (0,-1), (-1,0), (0,1)]

def heuristic(state, goal):
    count = 0
    for i in range(n):
        for j in range(n):
            if state[i][j] != 0 and state[i][j] != goal[i][j]:
                count += 1
    return count

def find_zero(state):
    for i in range(n):
        for j in range(n):
            if state[i][j] == 0:
                return i, j

def print_state(state):
    for row in state:
        print(*row)
    print()

def print_path(node):
    path = []
    while node:
        path.append(node["state"])
        node = node["parent"]
    path.reverse()
    for step in path:
        print_state(step)

def solve(initial, goal):
    counter = 0

    start = {
        "state": initial,
        "g": 0,
        "h": heuristic(initial, goal),
        "parent": None
    }
    start["f"] = start["g"] + start["h"]

    pq = []
    heappush(pq, (start["f"], counter, start))

    visited = set()

    while pq:
        _, _, current = heappop(pq)

        if current["h"] == 0:
            print_path(current)
            return

        visited.add(str(current["state"]))

        x, y = find_zero(current["state"])

        for dx, dy in moves:
            nx, ny = x + dx, y + dy

            if 0 <= nx < n and 0 <= ny < n:
                new_state = copy.deepcopy(current["state"])

                new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]

                if str(new_state) in visited:
                    continue

                counter += 1

                new_node = {
                    "state": new_state,
                    "g": current["g"] + 1,
                    "h": heuristic(new_state, goal),
                    "parent": current
                }
                new_node["f"] = new_node["g"] + new_node["h"]

                heappush(pq, (new_node["f"], counter, new_node))

    print("Không có lời giải!")

initial = [
    [2, 8, 3],
    [1, 6, 4],
    [7, 0, 5]
]

goal = [
    [1, 2, 3],
    [8, 0, 4],
    [7, 6, 5]
]

solve(initial, goal)

2 8 3
1 6 4
7 0 5

2 8 3
1 0 4
7 6 5

2 0 3
1 8 4
7 6 5

0 2 3
1 8 4
7 6 5

1 2 3
0 8 4
7 6 5

1 2 3
8 0 4
7 6 5



In [ ]:
# Bài 1

import copy
from heapq import heappush, heappop

n = 4

dx = [1, 0, -1, 0]
dy = [0, -1, 0, 1]

class Node:
    def __init__(self, matrix, empty_pos, g, parent):
        self.matrix = matrix
        self.empty_pos = empty_pos
        self.g = g
        self.h = 0
        self.f = 0
        self.parent = parent

    def __lt__(self, other):
        return self.f < other.f

def heuristic(matrix, goal):
    count = 0
    for i in range(n):
        for j in range(n):
            if matrix[i][j] != 0 and matrix[i][j] != goal[i][j]:
                count += 1
    return count

def is_valid(x, y):
    return 0 <= x < n and 0 <= y < n

def print_matrix(matrix):
    for row in matrix:
        print(row)
    print()

def print_path(node):
    if node is None:
        return
    print_path(node.parent)
    print_matrix(node.matrix)

def solve(initial, goal, empty_pos):

    open_list = []

    start = Node(initial, empty_pos, 0, None)
    start.h = heuristic(initial, goal)
    start.f = start.g + start.h

    heappush(open_list, start)

    visited = set()

    while open_list:
        current = heappop(open_list)

        if current.h == 0:
            print_path(current)
            return

        visited.add(str(current.matrix))

        x, y = current.empty_pos

        for i in range(4):
            new_x = x + dx[i]
            new_y = y + dy[i]

            if is_valid(new_x, new_y):
                new_matrix = copy.deepcopy(current.matrix)

                new_matrix[x][y], new_matrix[new_x][new_y] = new_matrix[new_x][new_y], new_matrix[x][y]

                if str(new_matrix) in visited:
                    continue

                child = Node(new_matrix, (new_x, new_y), current.g + 1, current)
                child.h = heuristic(new_matrix, goal)
                child.f = child.g + child.h

                heappush(open_list, child)

    print("Không tìm thấy lời giải!")

initial = [
    [1, 2, 3, 4],
    [5, 6, 0, 8],
    [9,10, 7,11],
    [13,14,15,12]
]

goal = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9,10,11,12],
    [13,14,15, 0]
]

empty_pos = (1, 2)

solve(initial, goal, empty_pos)

[1, 2, 3, 4]
[5, 6, 0, 8]
[9, 10, 7, 11]
[13, 14, 15, 12]

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 0, 11]
[13, 14, 15, 12]

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 0]
[13, 14, 15, 12]

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]

